# Prepare 91-feature relative position and velocity forecasts

Run notebook **32_continued_1v1.ipynb** to completion first. This notebook ingests its two-perspective NN-duel corpus and prepares supervised data for forecasting the opposing aircraft relative to the observer at **+1, +2 and +5 seconds**. It prepares data only; it does not collect flights or train a model.

Every input frame contains exactly the **91 policy features used by the neural pilots in notebooks 30 and 31**: 47 compact aircraft/friendly-missile values, 40 hostile-missile values and four energy values. Nothing is appended: no acceleration, selected skill, pilot identity, outcome or future state. Histories contain the pilots' chronological **20 frames at 10 Hz**.

Each physical target is `[opponent_position - observer_position, opponent_velocity - observer_velocity]` in world North/West/Up at the future timestamp. `targets.npy` stores that six-vector in metres and metres/second. `residuals.npy` subtracts a constant-relative-velocity baseline, and the matching baseline and per-horizon masks are saved separately. This makes the prepared data usable for direct regression or stable residual regression without regenerating the corpus.

In [1]:
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_RELATIVE_MOTION_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((path.resolve() for path in candidates
             if (path / "bvr_behavior_prediction/forecasting/relative_motion.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_RELATIVE_MOTION_ROOT to the Trajectory_Classification checkout.")
sys.path.insert(0, str(ROOT))

from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()  # Report the active kernel and native-library repair guidance.
import numpy as np
import pandas as pd
from IPython.display import display
from bvr_behavior_prediction.forecasting import relative_motion as motion
from bvr_behavior_prediction.forecasting.relative_motion import (
    HORIZONS, HISTORY_STEPS, RelativeMotionData, analyze_corpus, discover_corpus,
    nonoverlap_audit, prepare_dataset, read_json, sha256, write_json,
)

EXPERIMENT_ROOT = ROOT / "artifacts/relative_motion_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
source_default = ROOT / "shared/corpora/nn-duels-varied-v1/vw3"
source_setting = os.environ.get("BVR_RELATIVE_MOTION_CORPUS", str(source_default))
SOURCE = discover_corpus(ROOT, source_setting)
SOURCE_DIGEST = sha256(SOURCE / "manifest.json")
PREPARATION_DIGEST = sha256(motion.__file__)
DATA_DEFAULT = (EXPERIMENT_ROOT / "data" /
                f"{SOURCE.name}_{SOURCE_DIGEST[:12]}_{PREPARATION_DIGEST[:8]}")
DATA = Path(os.environ.get("BVR_RELATIVE_MOTION_DATA", str(DATA_DEFAULT))).expanduser().resolve()
SPLIT_SEED = 1729
print({"source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST,
       "prepared_output": str(DATA), "features_per_frame": 91,
       "history_steps": HISTORY_STEPS, "horizons_s": HORIZONS.tolist()})

{'source': 'C:\\Users\\theon\\Trajectory_Classification\\shared\\corpora\\nn-duels-varied-v1\\vw3', 'source_manifest_sha256': '145be1e59986135f7c0a97c5903447a094f59f54aace6b87a005c0c4c7a17310', 'prepared_output': 'C:\\Users\\theon\\Trajectory_Classification\\artifacts\\relative_motion_forecasting\\data\\vw3_145be1e59986_55ab4f59', 'features_per_frame': 91, 'history_steps': 20, 'horizons_s': [1.0, 2.0, 5.0]}


## Alignment, split and leakage contract

Both aircraft perspectives of a physical flight remain in the same partition through `scenario_group`. The seeded split is 40% train, 30% test and 30% validation, stratified by initial-range quantiles. Training histories are thinned onto adjacent two-second slots and alternate A01/B01, so no raw 20-frame input interval overlaps another retained training interval on the physical-flight timeline. Test and validation retain dense one-second origins.

A history and its label are eligible only while both aircraft are alive. Each horizon is masked independently when its exact future frame is missing or either aircraft is dead. Labels come only from the physical `tracks` table; target decisions are never read. Input feature scaling and residual-target scaling are fitted only on retained training histories/labels. The raw 91-channel frame array remains unchanged and memory-mapped.

The loader checks corpus hashes, source roles, timestamp cadence, the compact observation geometry against physical tracks, split isolation, target/baseline consistency and cross-perspective training non-overlap. Choose a new data directory whenever the source manifest, preparation code or split seed changes.

In [2]:
analysis = analyze_corpus(SOURCE)
episodes = analysis["episodes"]
eligible = analysis["eligible_by_horizon"]
print(f"Surveyed {len(episodes):,} perspectives from {episodes.episode_id.nunique():,} physical flights.")
display(episodes.groupby("observer_id").agg(
    perspectives=("episode_id", "size"), physical_flights=("episode_id", "nunique"),
    mean_frames=("frames", "mean")))
display(pd.DataFrame({"horizon_s": HORIZONS,
                      "eligible_dense_labels": eligible.sum(axis=0)}))

Read part-00000: 200 perspective episodes
Read part-00001: 200 perspective episodes
Read part-00002: 200 perspective episodes
Read part-00003: 200 perspective episodes
Read part-00004: 200 perspective episodes
Read part-00000: 200 perspective episodes
Read part-00001: 200 perspective episodes
Read part-00002: 200 perspective episodes
Read part-00003: 200 perspective episodes
Read part-00004: 200 perspective episodes
Read part-00005: 200 perspective episodes
Read part-00006: 200 perspective episodes
Read part-00007: 200 perspective episodes
Read part-00008: 200 perspective episodes
Read part-00009: 200 perspective episodes
Read part-00010: 200 perspective episodes
Read part-00011: 200 perspective episodes
Read part-00012: 200 perspective episodes
Read part-00013: 200 perspective episodes
Read part-00014: 200 perspective episodes
Read part-00015: 200 perspective episodes
Read part-00016: 200 perspective episodes
Read part-00017: 200 perspective episodes
Read part-00018: 200 perspective e

,perspectives,physical_flights,mean_frames
observer_id,,,
A01,10000,10000,1999.3026
B01,10000,10000,1999.3026


,horizon_s,eligible_dense_labels
0,1.0,3951872
1,2.0,3931872
2,5.0,3871872


In [3]:
if (DATA / "manifest.json").exists():
    manifest = read_json(DATA / "manifest.json")
    if manifest["source_manifest_sha256"] != SOURCE_DIGEST:
        raise ValueError("Existing data belongs to another corpus; choose a new output path.")
    if manifest["source_code_sha256"] != PREPARATION_DIGEST:
        raise ValueError("Preparation code changed; choose a new output path.")
    if manifest["split_seed"] != SPLIT_SEED:
        raise ValueError("Split seed changed; choose a new output path.")
    print("Reusing verified prepared data:", DATA)
else:
    manifest = prepare_dataset(ROOT, DATA, SOURCE, split_seed=SPLIT_SEED, analysis=analysis)

data = RelativeMotionData(DATA)
assert data.features.shape[1] == 91
assert data.manifest["features"] == analysis["features"]
display(pd.DataFrame(manifest["counts"]).T)
display(pd.Series({"raw_frames": len(data.features), "features_per_frame": 91,
                   "feature_array_GiB": data.features.nbytes / 2**30,
                   "prepared_windows": len(data.samples)}))
display(pd.Series(nonoverlap_audit(data.samples)))
display(pd.read_csv(DATA / "target_support_by_split.csv"))

Read part-00000: 200 perspective episodes
Read part-00001: 200 perspective episodes
Read part-00002: 200 perspective episodes
Read part-00003: 200 perspective episodes
Read part-00004: 200 perspective episodes
Read part-00005: 200 perspective episodes
Read part-00006: 200 perspective episodes
Read part-00007: 200 perspective episodes
Read part-00008: 200 perspective episodes
Read part-00009: 200 perspective episodes
Read part-00010: 200 perspective episodes
Read part-00011: 200 perspective episodes
Read part-00012: 200 perspective episodes
Read part-00013: 200 perspective episodes
Read part-00014: 200 perspective episodes
Read part-00015: 200 perspective episodes
Read part-00016: 200 perspective episodes
Read part-00017: 200 perspective episodes
Read part-00018: 200 perspective episodes
Read part-00019: 200 perspective episodes
Read part-00020: 200 perspective episodes
Read part-00021: 200 perspective episodes
Read part-00022: 200 perspective episodes
Read part-00023: 200 perspective e

,groups,perspective_episodes,physical_flights,windows,valid_by_horizon
train,4000,8000,4000,396556,"[396556, 395692, 388556]"
test,3000,6000,3000,1167032,"[1167032, 1161032, 1143032]"
validation,3000,6000,3000,1200344,"[1200344, 1194344, 1176344]"


raw_frames            3.998605e+07
features_per_frame    9.100000e+01
feature_array_GiB     1.355533e+01
prepared_windows      2.763932e+06
dtype: float64

verified                                                           True
training_intervals                                               396556
history_steps                                                        20
minimum_gap_frames                                                    0
scope                 both observer perspectives on each physical fl...
dtype: object

,split,horizon_s,windows,physical_flights,scenario_groups
0,train,1.0,396556,4000,4000
1,train,2.0,395692,4000,4000
2,train,5.0,388556,4000,4000
3,test,1.0,1167032,3000,3000
4,test,2.0,1161032,3000,3000
5,test,5.0,1143032,3000,3000
6,validation,1.0,1200344,3000,3000
7,validation,2.0,1194344,3000,3000
8,validation,5.0,1176344,3000,3000


## Inspect an exact training example

The example below shows the source feature order, a raw unstandardised policy history and the three future physical targets. `target = constant-relative-velocity baseline + residual` is checked at every valid horizon. The final receipt is written only after all integrity checks pass.

In [4]:
schema = read_json(SOURCE / "schema.json")
display(pd.DataFrame(schema["features"]))
sample_id = int(data.indices["train"][0])
sample = data.samples.iloc[sample_id]
history = data.history([sample_id], standardize=False)[0]
target = data.targets[sample_id]
baseline = data.baseline[sample_id]
residual = data.residuals[sample_id]
valid = data.valid[sample_id]
np.testing.assert_allclose(target[valid], (baseline + residual)[valid], rtol=1e-5, atol=1e-3)
display(sample)
display(pd.DataFrame({
    "history_step": np.arange(int(sample.step) - HISTORY_STEPS + 1, int(sample.step) + 1),
    "relative_n_m": history[:, 9] * 50000,
    "relative_w_m": history[:, 10] * 50000,
    "relative_u_m": history[:, 11] * 10000,
    "relative_vn_mps": history[:, 12] * 600,
    "relative_vw_mps": history[:, 13] * 600,
    "relative_vu_mps": history[:, 14] * 600,
}))
rows = []
for index, horizon in enumerate(HORIZONS):
    rows.append({"horizon_s": float(horizon), "valid": bool(valid[index]),
                 **dict(zip(data.manifest["target_fields"], target[index])),
                 **{f"residual_{name}": value for name, value in
                    zip(data.manifest["target_fields"], residual[index])}})
display(pd.DataFrame(rows))

write_json(EXPERIMENT_ROOT / "latest_dataset.json", {
    "directory": str(DATA), "manifest_sha256": sha256(DATA / "manifest.json"),
    "source": str(SOURCE), "source_manifest_sha256": SOURCE_DIGEST,
})
print("Relative-motion training data is ready:", DATA)

,column,name,divisor,formula
0,obs_00,own_north,50000.0,NaN
1,obs_01,own_west,50000.0,NaN
2,obs_02,own_up,10000.0,NaN
3,obs_03,own_velocity_north,600.0,NaN
4,obs_04,own_velocity_west,600.0,NaN
...,...,...,...,...
86,obs_86,hostile_3_seeker_active,1.0,NaN
87,obs_87,own_kinetic_energy,NaN,(speed_mps / 400)^2
88,obs_88,own_potential_energy,NaN,altitude_m / 12000
89,obs_89,target_kinetic_energy,NaN,(speed_mps / 400)^2


episode_id        nn_800000000_00000
observer_id                      A01
target_id                        B01
pilot_id                   pilot_006
scenario_group    nn_800000000_00000
split                          train
step                              20
time_s                           2.0
Name: 0, dtype: object

,history_step,relative_n_m,relative_w_m,relative_u_m,relative_vn_mps,relative_vw_mps,relative_vu_mps
0,1,28118.341797,9083.267578,-4376.736328,534.944397,16.137175,0.151665
1,2,28171.832031,9084.879883,-4376.677734,534.862244,16.133408,0.970950
2,3,28225.312500,9086.498047,-4376.498047,534.745239,16.217360,2.380244
3,4,28278.777344,9088.128906,-4376.182129,534.603638,16.385365,3.653972
4,5,28332.226562,9089.780273,-4375.753418,534.428406,16.610720,4.662149
5,6,28385.656250,9091.456055,-4375.242676,534.225647,16.852819,5.369148
6,7,28439.072266,9093.155273,-4374.678223,534.054321,17.074499,5.795706
7,8,28492.468750,9094.873047,-4374.085449,533.922974,17.237041,5.997025
8,9,28545.853516,9096.603516,-4373.479980,533.827209,17.356192,6.079587
9,10,28599.232422,9098.345703,-4372.870117,533.756592,17.453777,6.104085


,horizon_s,valid,relative_north_m,relative_west_m,relative_up_m,relative_vn_mps,relative_vw_mps,relative_vu_mps,residual_relative_north_m,residual_relative_west_m,residual_relative_up_m,residual_relative_vn_mps,residual_relative_vw_mps,residual_relative_vu_mps
0,1.0,True,29665.832031,9133.739258,-4361.663574,532.600830,14.101563,7.363829,-0.397697,-0.384336,-0.021820,-0.848194,-3.839221,2.007985
1,2.0,True,30197.189453,9144.608398,-4351.118164,529.489380,9.274284,12.289255,-2.488786,-7.456166,5.168166,-3.959588,-8.666501,6.933412
2,5.0,True,31741.859375,9178.286133,-4327.311523,495.429901,16.727959,2.375839,-58.166180,-27.600988,12.907105,-38.019104,-1.212826,-2.980004


Relative-motion training data is ready: C:\Users\theon\Trajectory_Classification\artifacts\relative_motion_forecasting\data\vw3_145be1e59986_55ab4f59
